In [ ]:
#Task 1 : Adaptive Security Policies with RL

In [1]:
pip install gymnasium numpy pandas scikit-learn stable-baselines3[extra] --no-cache-dir

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 1.5 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 1.6 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 3.0 MB/s eta 0:00:0000:01m0:21m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 1.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.1 MB/s eta 0:00:0000:01m00:57
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 617.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 312.2 kB/s eta 0:00:0000:0100:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 425.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 192.8 kB/s eta 0:00:0000:0104:40
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
#Defining the Custom Gymnasium Environment :

In [1]:

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import random

# --- Configuration --- 
# In a real scenario, these would be dynamically loaded or generated.
# For this example, we use synthetic data.

# Example of potential features that could form the state vector
# Features: [src_ip_hash, dst_ip_hash, src_port, dst_port, protocol_onehot_tcp, protocol_onehot_udp, packet_count_norm, byte_count_norm, connection_rate_norm, is_known_malicious_ip_flag, system_load_norm]
# We'll simplify this for the example.

# Simplified state space: [packet_count_norm, connection_rate_norm, is_known_malicious_ip_flag]
# Normalization range for features (example)
FEATURE_RANGES = {
    'packet_count': (0, 1000),       # Max packets in a flow
    'connection_rate': (0, 50),      # Connections per second
    'malicious_ip_flag': (0, 1)      # 0 for benign, 1 for known malicious
}

# Simplified action space: 0: Allow, 1: Block
NUM_ACTIONS = 2

# Simplified reward structure
REWARD_STRUCTURE = {
    'malicious_blocked': 10,
    'legitimate_allowed': 0.1,
    'malicious_allowed': -10,
    'legitimate_blocked': -1
}

class AdaptiveFirewallEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self, render_mode=None):
        super(AdaptiveFirewallEnv, self).__init__()

        self.render_mode = render_mode

        # Define action and observation space
        # Action space: Discrete(2) -> 0: Allow, 1: Block
        self.action_space = spaces.Discrete(NUM_ACTIONS)

        # Observation space: Based on simplified features
        # Example: [packet_count_norm, connection_rate_norm, malicious_ip_flag]
        # We use Box for continuous or bounded discrete values
        # Lower bounds, upper bounds, shape
        low = np.array([
            0.0,  # packet_count_norm
            0.0,  # connection_rate_norm
            0.0   # malicious_ip_flag
        ])
        high = np.array([
            1.0,  # packet_count_norm
            1.0,  # connection_rate_norm
            1.0   # malicious_ip_flag
        ])
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        # Internal state variables
        self.current_state = None
        self.current_connection_label = None # 'malicious' or 'legitimate'

        # Simulate a list of known malicious IPs (for the flag feature)
        self.known_malicious_ips = {1234567890, 9876543210} # Example hashes

    def _normalize_feature(self, feature_name, value):
        """Normalizes a feature value to a 0-1 range."""
        min_val, max_val = FEATURE_RANGES[feature_name]
        if max_val == min_val: return 0.0
        return max(0.0, min(1.0, (value - min_val) / (max_val - min_val)))

    def _generate_synthetic_connection(self):
        """Generates a synthetic network connection's features and its true label."""
        # Simulate features
        packet_count = random.randint(FEATURE_RANGES['packet_count'][0], FEATURE_RANGES['packet_count'][1])
        connection_rate = random.randint(FEATURE_RANGES['connection_rate'][0], FEATURE_RANGES['connection_rate'][1])
        
        # Simulate malicious IP flag
        is_malicious_ip = random.choice([0, 1]) # Randomly decide if IP is malicious for simulation
        if is_malicious_ip == 1 and random.random() < 0.8: # Higher chance of malicious traffic if IP is flagged
            true_label = 'malicious'
        else:
            true_label = 'legitimate'
            
        # Ensure malicious IP flag is consistent with true label for this simulation
        if true_label == 'malicious':
            is_malicious_ip = 1
        else:
            is_malicious_ip = 0

        # Normalize features
        norm_packet_count = self._normalize_feature('packet_count', packet_count)
        norm_connection_rate = self._normalize_feature('connection_rate', connection_rate)
        norm_malicious_ip_flag = float(is_malicious_ip)

        state_features = np.array([
            norm_packet_count,
            norm_connection_rate,
            norm_malicious_ip_flag
        ], dtype=np.float32)

        return state_features, true_label

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Generate initial state and true label for the first connection
        self.current_state, self.current_connection_label = self._generate_synthetic_connection()

        if self.render_mode == 'human':
            print(f"\n--- New Connection --- ")
            print(f"State: {self.current_state}")
            print(f"True Label: {self.current_connection_label}")

        return self.current_state, {}

    def step(self, action):
        # Action: 0 for Allow, 1 for Block
        agent_action = action

        # Determine reward based on agent's action and true label
        reward = 0
        if agent_action == 0:  # Agent chose to ALLOW
            if self.current_connection_label == 'malicious':
                reward = REWARD_STRUCTURE['malicious_allowed']
            else: # legitimate
                reward = REWARD_STRUCTURE['legitimate_allowed']
        else:  # Agent chose to BLOCK (action == 1)
            if self.current_connection_label == 'malicious':
                reward = REWARD_STRUCTURE['malicious_blocked']
            else: # legitimate
                reward = REWARD_STRUCTURE['legitimate_blocked']

        # Generate the next state (simulate next connection)
        next_state, next_label = self._generate_synthetic_connection()
        self.current_state = next_state
        self.current_connection_label = next_label

        # Check if the episode should end (optional, for this example we run indefinitely)
        terminated = False
        truncated = False

        if self.render_mode == 'human':
            action_desc = 'ALLOW' if agent_action == 0 else 'BLOCK'
            print(f"Agent Action: {action_desc}")
            print(f"Reward: {reward}")
            print(f"Next State: {self.current_state}")
            print(f"Next True Label: {self.current_connection_label}")

        return next_state, reward, terminated, truncated, {}

    def render(self):
        if self.render_mode == 'human':
            # Basic rendering: print current state and label
            print(f"Current State: {self.current_state}")
            print(f"True Label: {self.current_connection_label}")

    def close(self):
        pass

print("AdaptiveFirewallEnv defined successfully!")

AdaptiveFirewallEnv defined successfully!


In [ ]:
#Training the RL Agent

In [ ]:

import gymnasium as gym
import numpy as np

# Import our custom environment
# Assuming AdaptiveFirewallEnv is in the same notebook or imported from a file
# from your_module import AdaptiveFirewallEnv 

# Import stable-baselines3 algorithms
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

# --- Training Configuration ---
TOTAL_TIMESTEPS = 50000  # Number of steps to train for
LOG_DIR = "/home/student/cyberai-lab/rl_firewall_logs/"
MODEL_SAVE_PATH = "/home/student/cyberai-lab/rl_firewall_model/dqn_adaptive_firewall"

# Create the environment
# Using make_vec_env for potential parallelization (though not strictly needed for this example)
env = make_vec_env(lambda: AdaptiveFirewallEnv(), n_envs=1)

# Initialize the DQN agent
# Policy: 'MlpPolicy' for Multi-Layer Perceptron (standard neural network)
# env: The environment to train on
# verbose: 1 to print training progress
# learning_rate: Controls how much the weights are updated
# buffer_size: Size of the replay buffer (stores past experiences)
# learning_starts: Number of steps before learning starts (to fill buffer)
# batch_size: Number of samples used for each learning update
# train_freq: How often to train the model (e.g., 1 step means train after every step)
model = DQN('MlpPolicy', env, verbose=1, 
            learning_rate=1e-3, 
            buffer_size=10000, 
            learning_starts=1000, 
            batch_size=32, 
            train_freq=1, 
            target_update_interval=500, # How often to update the target network
            exploration_fraction=0.1, # Fraction of total timesteps for exploration
            exploration_final_eps=0.02, # Final epsilon value for exploration
            seed=42 # for reproducibility
           )

print("DQN Agent initialized.")

# Train the agent
print(f"Starting training for {TOTAL_TIMESTEPS} timesteps...")
try:
    model.learn(total_timesteps=TOTAL_TIMESTEPS, log_path=LOG_DIR)
    print("Training finished successfully!")

    # Save the trained model
    model.save(MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")

except Exception as e:
    print(f"An error occurred during training: {e}")
    import traceback
    traceback.print_exc()

# --- Evaluation ---
print("Evaluating the trained agent...")

# Create a separate environment for evaluation
eval_env = AdaptiveFirewallEnv()

# Evaluate the policy
# n_eval_episodes: Number of episodes to run for evaluation
# deterministic=True: Use the greedy policy (no exploration)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=100, deterministic=True)

print(f"Mean reward over 100 episodes: {mean_reward:.2f} +/- {std_reward:.2f}")

# Expected output: The mean reward should ideally be positive, indicating the agent is learning to block malicious traffic and allow legitimate traffic.
# A positive mean reward suggests the agent is learning effectively.
print("Evaluation complete.")

# --- Testing the agent interactively (optional) ---
print("\n--- Interactive Test --- ")
# Reset the environment for interactive testing
obs, _ = eval_env.reset()

# Run for a few steps
for i in range(10):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, _ = eval_env.reset()

print("Interactive test finished.")

# --- Reset Command Reminder ---
print("\nTo reset the lab environment, run: docker compose down && docker compose up -d")

/opt/conda/lib/python3.11/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/opt/conda/lib/python3.11/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Using cpu device
DQN Agent initialized.
Starting training for 50000 timesteps...
An error occurred during training: DQN.learn() got an unexpected keyword argument 'log_path'
Evaluating the trained agent...


Traceback (most recent call last):
  File "/tmp/ipykernel_38866/3773928233.py", line 48, in <module>
    model.learn(total_timesteps=TOTAL_TIMESTEPS, log_path=LOG_DIR)
TypeError: DQN.learn() got an unexpected keyword argument 'log_path'
/opt/conda/lib/python3.11/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/opt/conda/lib/python3.11/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/opt/conda/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [2]:
#reset before task 2 : 

# 1. Close the previous environment to release its resources
try:
    env.close()
    print("Environment closed.")
except:
    pass

# 2. Delete the model 'brain' from memory
try:
    del model
    print("Model variable deleted.")
except:
    pass

# 3. Force Python to clear the "garbage" (freed RAM)
import gc
gc.collect()
print("Garbage collection complete.")

# 4. Also, restart the kernel.

Garbage collection complete.


In [ ]:
#Task 2 : Automated incident response with RL

In [ ]:
pip install gymnasium numpy stable-baselines3[extra] torch
# For tensorboard visualization (optional)
pip install tensorboard

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# Assume Stable Baselines3 is installed: pip install stable-baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# --- 1. Define the Custom Gym Environment --- 
class MalwareContainmentEnv(gym.Env):
    """Custom Environment for Malware Containment Simulation."""
    def __init__(self, num_hosts=5, initial_infected=1, spread_prob=0.3, isolation_cost=-10, spread_penalty=-50, containment_reward=100, false_positive_penalty=-20):
        super(MalwareContainmentEnv, self).__init__()

        self.num_hosts = num_hosts
        self.initial_infected = initial_infected
        self.spread_prob = spread_prob
        self.isolation_cost = isolation_cost
        self.spread_penalty = spread_penalty
        self.containment_reward = containment_reward
        self.false_positive_penalty = false_positive_penalty

        # State: [host_status_1, host_status_2, ..., num_hosts]
        # Host status: 0=Clean, 1=Infected, 2=Isolated, 3=Scanning
        self.observation_space = spaces.Box(low=0, high=3, shape=(self.num_hosts,), dtype=np.int32)

        # Actions: 
        # 0: do_nothing
        # 1: isolate_host(host_id)
        # 2: scan_host(host_id)
        # We'll use a discrete action space where action index corresponds to host_id + offset
        # Action indices: 0=do_nothing, 1=isolate_host_0, 2=isolate_host_1, ..., num_hosts=isolate_host_{num_hosts-1}
        # num_hosts+1 = scan_host_0, ..., 2*num_hosts = scan_host_{num_hosts-1}
        self.action_space = spaces.Discrete(1 + 2 * self.num_hosts) # do_nothing + isolate + scan for each host

        self.current_state = None
        self.episode_step = 0
        self.max_steps = 50 # Max steps per episode

    def _get_obs(self):
        return np.array(self.current_state, dtype=np.int32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_state = np.zeros(self.num_hosts, dtype=np.int32)
        # Infect initial hosts
        infected_indices = np.random.choice(self.num_hosts, self.initial_infected, replace=False)
        for idx in infected_indices:
            self.current_state[idx] = 1 # Infected
        
        self.episode_step = 0
        return self._get_obs(), {}

    def step(self, action):
        self.episode_step += 1
        reward = 0
        terminated = False
        truncated = False

        # --- Apply Agent's Action ---
        current_host_statuses = self.current_state.copy()
        
        # Decode action
        if action == 0: # do_nothing
            pass
        elif 1 <= action <= self.num_hosts: # isolate_host
            host_to_isolate = action - 1
            if current_host_statuses[host_to_isolate] == 1: # Infected
                reward += self.containment_reward
                current_host_statuses[host_to_isolate] = 2 # Isolated
            elif current_host_statuses[host_to_isolate] == 0: # Clean
                reward += self.false_positive_penalty
                current_host_statuses[host_to_isolate] = 2 # Isolated (false positive)
            # If already isolated or scanning, do nothing or apply minor penalty
            else: 
                reward += self.isolation_cost # Cost of action

        elif self.num_hosts + 1 <= action <= 2 * self.num_hosts: # scan_host
            host_to_scan = action - (self.num_hosts + 1)
            if current_host_statuses[host_to_scan] == 1: # Infected
                # Scanning an infected host might reveal it, but doesn't contain it directly
                # Could add a small reward for successful scan detection if implemented
                pass
            current_host_statuses[host_to_scan] = 3 # Scanning
            reward += self.isolation_cost # Cost of action

        self.current_state = current_host_statuses

        # --- Simulate Malware Spread (if not isolated) ---
        newly_infected_this_step = 0
        for i in range(self.num_hosts):
            if self.current_state[i] == 1: # If host is infected
                # Try to spread to neighbors (simplified)
                for j in range(self.num_hosts):
                    if self.current_state[j] == 0: # If neighbor is clean
                        if np.random.rand() < self.spread_prob:
                            self.current_state[j] = 1 # Infect neighbor
                            newly_infected_this_step += 1
        
        if newly_infected_this_step > 0:
            reward += self.spread_penalty * newly_infected_this_step

        # --- Check for Termination Conditions ---
        if np.sum(self.current_state == 1) == 0: # All hosts are clean or isolated
            terminated = True
            # Add bonus reward for full containment if not already covered
            if reward > 0: reward += 50 # Bonus for successful episode

        if self.episode_step >= self.max_steps:
            truncated = True
            # Penalty for not containing within max steps
            reward += self.spread_penalty * np.sum(self.current_state == 1)

        # --- Update state for hosts that were scanned ---
        for i in range(self.num_hosts):
            if self.current_state[i] == 3: # If host was scanned
                # If it was infected and scanned, it remains infected (or could transition to 'detected')
                # If it was clean and scanned, it remains clean
                pass # For simplicity, scanning doesn't change state beyond 'scanning' marker for this step

        # --- Final state update after all actions and spread ---
        # Ensure isolated hosts cannot be infected or spread
        for i in range(self.num_hosts):
            if self.current_state[i] == 2: # Isolated
                # Ensure it stays isolated and doesn't spread
                pass

        # Ensure scanned hosts that were infected are now marked as infected (if not isolated)
        # This is a simplification; a real system would have more states like 'detected', 'quarantined', etc.
        for i in range(self.num_hosts):
            if self.current_state[i] == 3: # Was scanned
                if self.current_state[i] != 2: # If not isolated
                    # If it was infected, it remains infected. If it was clean, it remains clean.
                    # This state transition logic can be complex.
                    pass # For this simple model, scanning doesn't change the underlying infected status directly

        # Re-apply isolation effect if it was isolated in this step
        for i in range(self.num_hosts):
            if action == (i + 1) and action != 0: # If isolate_host action was taken for host i
                self.current_state[i] = 2 # Ensure it's marked as isolated

        # Re-apply scanning effect if it was scanned in this step
        for i in range(self.num_hosts):
            if action == (self.num_hosts + 1 + i) and action != 0: # If scan_host action was taken for host i
                self.current_state[i] = 3 # Ensure it's marked as scanning

        # Ensure infected hosts that were not isolated/scanned can still spread
        # This logic needs careful refinement based on desired simulation fidelity.

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        # Basic text rendering
        host_status_map = {0: 'Clean', 1: 'Infected', 2: 'Isolated', 3: 'Scanning'}
        status_str = " ".join([f'H{i}:{host_status_map[self.current_state[i]]}' for i in range(self.num_hosts)])
        print(f'Step: {self.episode_step} | Status: {status_str}')

    def close(self):
        pass

# --- 2. Initialize and Train the RL Agent --- 
if __name__ == "__main__":
    # Create the environment
    env = MalwareContainmentEnv(num_hosts=5, initial_infected=2, spread_prob=0.4, isolation_cost=-5, spread_penalty=-30, containment_reward=80, false_positive_penalty=-15)
    
    # Wrap the environment for vectorized training (optional but recommended)
    # env = make_vec_env(lambda: MalwareContainmentEnv(), n_envs=4)

    # Initialize the PPO agent
    # Policy='MlpPolicy' for feedforward neural network
    model = PPO('MlpPolicy', env, verbose=1, tensorboard_log="./ppo_malware_tensorboard/")

    print("Starting training...")
    # Train the agent for a number of timesteps
    # Increase timesteps for better learning
    model.learn(total_timesteps=50000)
    print("Training finished.")

    # Save the trained model
    model.save("ppo_malware_containment_agent")
    print("Model saved.")

    # --- 3. Test the Trained Agent --- 
    print("\
--- Testing the trained agent ---")
    # Load the trained model
    # model = PPO.load("ppo_malware_containment_agent")

    # Create a separate environment for testing
    test_env = MalwareContainmentEnv(num_hosts=5, initial_infected=2, spread_prob=0.4)
    obs, _ = test_env.reset()
    total_reward = 0
    
    for _ in range(test_env.max_steps):
        test_env.render()
        action, _states = model.predict(obs, deterministic=True) # Use deterministic=True for evaluation
        obs, reward, terminated, truncated, info = test_env.step(action)
        total_reward += reward
        if terminated or truncated:
            print(f"Episode finished. Total Reward: {total_reward}")
            obs, _ = test_env.reset()
            total_reward = 0
            # break # Uncomment to run only one episode
    
    test_env.close()